# 08 RAG Answer Generation

## Goal

This notebook turns retrieved SEC evidence into grounded answers with citations.

The workflow is:

```text
user question
→ retrieve evidence
→ build prompt
→ send prompt to local LLM
→ generate grounded answer
→ attach citations
→ save answer examples
```

This notebook uses local Ollama for answer generation.

The key rule:

```text
The model must answer only from retrieved SEC filing evidence.
```

If the evidence does not support an answer, the system should say that the filings did not provide enough evidence.

In [25]:
# Import tools for file and folder paths
from pathlib import Path

# Import pandas for working with tables
import pandas as pd

# Import numpy for numerical work
import numpy as np

# Import regular expressions for text cleaning and tokenization
import re

# Import ChromaDB for vector search
import chromadb

# Import sentence-transformers for embeddings and reranking
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder

# Import BM25 keyword search
from rank_bm25 import BM25Okapi

# Import requests for calling local Ollama
import requests

# Import json for working with API responses
import json

# Import textwrap for clean text display
import textwrap

# Import time for timing retrieval and generation
import time

In [26]:
# Detect the project root automatically
# If this notebook is inside the notebooks folder, move one level up
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Create main project paths
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
VECTORSTORE_DIR = DATA_DIR / "vectorstore"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Set Chroma vector database folder
CHROMA_DIR = VECTORSTORE_DIR / "chroma_sec_10k"

# Create report folder if needed
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Print paths to confirm everything is correct
print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)
print("Vectorstore folder:", VECTORSTORE_DIR)
print("Reports folder:", REPORTS_DIR)
print("Chroma folder:", CHROMA_DIR)

Project root: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI
Processed data folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed
Vectorstore folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\vectorstore
Reports folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\reports
Chroma folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\vectorstore\chroma_sec_10k


#### Load RAG chunks

In [27]:
# Set path to final RAG chunks from notebook 04
rag_chunks_file = PROCESSED_DIR / "sec_10k_rag_chunks.csv"

# Check that the chunk file exists
if not rag_chunks_file.exists():
    raise FileNotFoundError(
        f"Could not find {rag_chunks_file}. Run 04_chunking_experiments.ipynb first."
    )

# Load RAG chunks
rag_chunks_df = pd.read_csv(rag_chunks_file)

# Fill missing values to avoid metadata errors
rag_chunks_df = rag_chunks_df.fillna("")

# Preview chunk dataset
rag_chunks_df.head()

,chunk_id,document_id,ticker,company_name,filing_date,accession_number,filing_url,section_name,source_label,citation_label,chunk_index,chunk_size,chunk_overlap,start_word,end_word,chunk_word_count,chunk_character_count,chunk_text
0,AAPL_2025-10-31_item_1_business_chunk_0000,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 0",0,350,75,0,350,350,2125,Item 1. Business Company Background The Compan...
1,AAPL_2025-10-31_item_1_business_chunk_0001,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 1",1,350,75,275,625,350,2341,Apple Inc. | 2025 Form 10-K | 1 Services Adver...
2,AAPL_2025-10-31_item_1_business_chunk_0002,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 2",2,350,75,550,900,350,2429,"Greater China includes China mainland, Hong Ko..."
3,AAPL_2025-10-31_item_1_business_chunk_0003,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 3",3,350,75,825,1175,350,2563,by imitating the Company’s products and infrin...
4,AAPL_2025-10-31_item_1_business_chunk_0004,AAPL_2025-10-31_item_1_business,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,item_1_business,AAPL 2025-10-31 10-K item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 4",4,350,75,1100,1450,350,2357,provide products and services at little or no ...


#### Load embedding summary

In [28]:
# Set path to embedding summary from notebook 05
embedding_summary_file = PROCESSED_DIR / "sec_10k_embedding_summary.csv"

# Check that the embedding summary exists
if not embedding_summary_file.exists():
    raise FileNotFoundError(
        f"Could not find {embedding_summary_file}. Run 05_embeddings_and_vector_store.ipynb first."
    )

# Load embedding summary
embedding_summary = pd.read_csv(embedding_summary_file)

# Convert embedding summary into a dictionary
embedding_settings = dict(
    zip(
        embedding_summary["setting"],
        embedding_summary["value"]
    )
)

# Pull settings created in notebook 05
EMBEDDING_MODEL_NAME = embedding_settings["embedding_model"]
COLLECTION_NAME = embedding_settings["collection_name"]

# Display settings
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Collection name:", COLLECTION_NAME)
print("Chroma path:", CHROMA_DIR)

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Collection name: sec_10k_rag_chunks
Chroma path: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\vectorstore\chroma_sec_10k


#### Load models and Chroma collection

In [29]:
# Load the same embedding model used to create the vector store
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Set reranker model name
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Load cross-encoder reranker
reranker_model = CrossEncoder(RERANKER_MODEL_NAME)

# Create persistent Chroma client
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

# Load existing Chroma collection
collection = chroma_client.get_collection(
    name=COLLECTION_NAME
)

# Print status
print("Embedding model loaded:", EMBEDDING_MODEL_NAME)
print("Reranker model loaded:", RERANKER_MODEL_NAME)
print("Collection name:", COLLECTION_NAME)
print("Records in collection:", collection.count())
print("Rows in chunk dataset:", len(rag_chunks_df))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
Reranker model loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
Collection name: sec_10k_rag_chunks
Records in collection: 520
Rows in chunk dataset: 520


#### Validate data and vectorstore

In [30]:
# Count records in Chroma
collection_count = collection.count()

# Count rows in chunk dataset
chunk_count = len(rag_chunks_df)

# Print counts
print("Chroma records:", collection_count)
print("Chunk rows:", chunk_count)

# Stop if counts do not match
if collection_count != chunk_count:
    raise ValueError("Chroma collection count does not match RAG chunk dataset count.")

# Confirm validation passed
print("Vectorstore validation passed.")

Chroma records: 520
Chunk rows: 520
Vectorstore validation passed.


Retrieval functions

## Retrieval Layer

This notebook reuses the retrieval logic from notebook 07.

The retriever combines:

```text
vector search
+ BM25 keyword search
+ reciprocal rank fusion
+ cross-encoder reranking
```

The output is citation-ready evidence that can be passed into the LLM.

#### Chroma filter helper

In [31]:
def build_chroma_where_filter(ticker=None, section_name=None):
    """
    Build a Chroma-compatible metadata filter.

    Chroma requires exactly one top-level filter condition.
    Multiple filters must be combined with $and.
    """

    # Create a list for filter conditions
    filter_conditions = []

    # Add ticker filter if provided
    if ticker is not None:
        filter_conditions.append({"ticker": ticker})

    # Add section filter if provided
    if section_name is not None:
        filter_conditions.append({"section_name": section_name})

    # Return no filter when no conditions exist
    if len(filter_conditions) == 0:
        return None

    # Return one filter directly
    if len(filter_conditions) == 1:
        return filter_conditions[0]

    # Combine multiple filters with Chroma's $and operator
    return {"$and": filter_conditions}

#### Vector search function

In [32]:
def vector_search(query, top_k=10, ticker=None, section_name=None):
    """
    Run semantic vector search against Chroma.
    """

    # Embed the query
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    # Build metadata filter
    where_filter = build_chroma_where_filter(
        ticker=ticker,
        section_name=section_name
    )

    # Query Chroma
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where=where_filter,
        include=["documents", "metadatas", "distances"]
    )

    # Create result records
    result_records = []

    # Loop through returned results
    for rank, (doc, metadata, distance) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ),
        start=1
    ):

        # Store one vector search result
        result_records.append({
            "chunk_id": metadata.get("chunk_id", ""),
            "vector_rank": rank,
            "vector_distance": distance,
            "ticker": metadata.get("ticker", ""),
            "company_name": metadata.get("company_name", ""),
            "filing_date": metadata.get("filing_date", ""),
            "section_name": metadata.get("section_name", ""),
            "citation_label": metadata.get("citation_label", ""),
            "filing_url": metadata.get("filing_url", ""),
            "chunk_text": doc
        })

    # Return results as DataFrame
    return pd.DataFrame(result_records)

BM25 tokenizer and search

In [33]:
def tokenize_text(text):
    """
    Convert text into simple lowercase tokens for BM25 keyword search.
    """

    # Convert text to string and lowercase it
    text = str(text).lower()

    # Keep only letters, numbers, and spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Split into tokens
    tokens = text.split()

    # Return tokens
    return tokens


def bm25_search(query, top_k=10, ticker=None, section_name=None):
    """
    Run BM25 keyword search over the chunk dataset.
    """

    # Create a filtered copy of the chunk dataset
    search_df = rag_chunks_df.copy()

    # Apply ticker filter if provided
    if ticker is not None:
        search_df = search_df[search_df["ticker"] == ticker].copy()

    # Apply section filter if provided
    if section_name is not None:
        search_df = search_df[search_df["section_name"] == section_name].copy()

    # Reset index after filtering
    search_df = search_df.reset_index(drop=True)

    # Return empty DataFrame if no records match filters
    if len(search_df) == 0:
        return pd.DataFrame()

    # Tokenize chunk texts
    tokenized_corpus = [
        tokenize_text(text)
        for text in search_df["chunk_text"].tolist()
    ]

    # Build BM25 index
    bm25 = BM25Okapi(tokenized_corpus)

    # Tokenize query
    tokenized_query = tokenize_text(query)

    # Get BM25 scores
    scores = bm25.get_scores(tokenized_query)

    # Add BM25 scores
    search_df["bm25_score"] = scores

    # Sort by BM25 score
    search_df = search_df.sort_values(
        "bm25_score",
        ascending=False
    ).head(top_k).copy()

    # Add BM25 rank
    search_df["bm25_rank"] = range(1, len(search_df) + 1)

    # Select output columns
    output_df = search_df[
        [
            "chunk_id",
            "bm25_rank",
            "bm25_score",
            "ticker",
            "company_name",
            "filing_date",
            "section_name",
            "citation_label",
            "filing_url",
            "chunk_text"
        ]
    ].copy()

    # Return BM25 results
    return output_df

Hybrid search

In [34]:
def hybrid_search(query, top_k=5, candidate_k=20, ticker=None, section_name=None, rrf_k=60):
    """
    Combine vector search and BM25 search using reciprocal rank fusion.
    """

    # Run vector search
    vector_df = vector_search(
        query=query,
        top_k=candidate_k,
        ticker=ticker,
        section_name=section_name
    )

    # Run BM25 search
    bm25_df = bm25_search(
        query=query,
        top_k=candidate_k,
        ticker=ticker,
        section_name=section_name
    )

    # Create dictionary to store merged results
    merged_records = {}

    # Add vector results
    for _, row in vector_df.iterrows():

        # Get chunk ID
        chunk_id = row["chunk_id"]

        # Initialize record if needed
        if chunk_id not in merged_records:
            merged_records[chunk_id] = {
                "chunk_id": chunk_id,
                "ticker": row["ticker"],
                "company_name": row["company_name"],
                "filing_date": row["filing_date"],
                "section_name": row["section_name"],
                "citation_label": row["citation_label"],
                "filing_url": row["filing_url"],
                "chunk_text": row["chunk_text"],
                "vector_rank": None,
                "vector_distance": None,
                "bm25_rank": None,
                "bm25_score": None,
                "rrf_score": 0
            }

        # Store vector rank and distance
        merged_records[chunk_id]["vector_rank"] = row["vector_rank"]
        merged_records[chunk_id]["vector_distance"] = row["vector_distance"]

        # Add RRF score from vector rank
        merged_records[chunk_id]["rrf_score"] += 1 / (rrf_k + row["vector_rank"])

    # Add BM25 results
    for _, row in bm25_df.iterrows():

        # Get chunk ID
        chunk_id = row["chunk_id"]

        # Initialize record if needed
        if chunk_id not in merged_records:
            merged_records[chunk_id] = {
                "chunk_id": chunk_id,
                "ticker": row["ticker"],
                "company_name": row["company_name"],
                "filing_date": row["filing_date"],
                "section_name": row["section_name"],
                "citation_label": row["citation_label"],
                "filing_url": row["filing_url"],
                "chunk_text": row["chunk_text"],
                "vector_rank": None,
                "vector_distance": None,
                "bm25_rank": None,
                "bm25_score": None,
                "rrf_score": 0
            }

        # Store BM25 rank and score
        merged_records[chunk_id]["bm25_rank"] = row["bm25_rank"]
        merged_records[chunk_id]["bm25_score"] = row["bm25_score"]

        # Add RRF score from BM25 rank
        merged_records[chunk_id]["rrf_score"] += 1 / (rrf_k + row["bm25_rank"])

    # Convert merged records to DataFrame
    hybrid_df = pd.DataFrame(list(merged_records.values()))

    # Return empty result safely
    if hybrid_df.empty:
        return pd.DataFrame()

    # Sort by RRF score
    hybrid_df = hybrid_df.sort_values(
        "rrf_score",
        ascending=False
    ).head(top_k).reset_index(drop=True)

    # Add hybrid rank
    hybrid_df["hybrid_rank"] = range(1, len(hybrid_df) + 1)

    # Return hybrid results
    return hybrid_df

Reranking and final retriever

In [35]:
def rerank_results(query, candidate_df, top_k=5):
    """
    Rerank candidate chunks using a cross-encoder.
    """

    # Return empty DataFrame if there are no candidates
    if candidate_df.empty:
        return pd.DataFrame()

    # Create query-document pairs
    pairs = [
        [query, chunk_text]
        for chunk_text in candidate_df["chunk_text"].tolist()
    ]

    # Predict relevance scores
    rerank_scores = reranker_model.predict(pairs)

    # Copy candidate results
    reranked_df = candidate_df.copy()

    # Add rerank scores
    reranked_df["rerank_score"] = rerank_scores

    # Sort by rerank score
    reranked_df = reranked_df.sort_values(
        "rerank_score",
        ascending=False
    ).head(top_k).reset_index(drop=True)

    # Add rerank rank
    reranked_df["rerank_rank"] = range(1, len(reranked_df) + 1)

    # Return reranked results
    return reranked_df


def retrieve_evidence(query, top_k=5, candidate_k=30, ticker=None, section_name=None, use_reranker=True):
    """
    Final evidence retriever.

    Steps:
    1. Hybrid search.
    2. Optional reranking.
    3. Return citation-ready evidence.
    """

    # Run hybrid search
    hybrid_candidates = hybrid_search(
        query=query,
        top_k=candidate_k,
        candidate_k=candidate_k,
        ticker=ticker,
        section_name=section_name
    )

    # Return empty DataFrame if no candidates exist
    if hybrid_candidates.empty:
        return pd.DataFrame()

    # Apply reranker if requested
    if use_reranker:
        final_results = rerank_results(
            query=query,
            candidate_df=hybrid_candidates,
            top_k=top_k
        )

    # Otherwise use hybrid search order
    else:
        final_results = hybrid_candidates.head(top_k).copy()

    # Return final evidence
    return final_results

Ollama answer generation

## Local LLM Answer Generation

This notebook uses Ollama for local answer generation.

The answer generator will:

```text
take the user question
+ retrieved SEC evidence
+ strict grounding instructions
→ produce a cited answer
```

The LLM is not allowed to make claims that are not supported by the retrieved evidence.

In [36]:
# Set local Ollama endpoint
OLLAMA_URL = "http://localhost:11434/api/generate"

# Set Ollama model name
# Change this to a model you have installed locally
OLLAMA_MODEL = "llama3.1:8b"

# Print settings
print("Ollama URL:", OLLAMA_URL)
print("Ollama model:", OLLAMA_MODEL)

Ollama URL: http://localhost:11434/api/generate
Ollama model: llama3.1:8b


In [37]:
def test_ollama_connection():
    """
    Test whether Ollama is running locally.
    """

    # Create a tiny test payload
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": "Reply with exactly: Ollama is working.",
        "stream": False
    }

    # Try to call Ollama
    try:
        response = requests.post(
            OLLAMA_URL,
            json=payload,
            timeout=60
        )

        # Print status code
        print("Status code:", response.status_code)

        # Raise error if request failed
        response.raise_for_status()

        # Parse response JSON
        result = response.json()

        # Print model response
        print(result.get("response", ""))

        # Return True if successful
        return True

    # Handle connection or model errors
    except Exception as error:
        print("Ollama connection failed.")
        print("Make sure Ollama is running and the model name is installed.")
        print("Error:", error)
        return False


# Run connection test
ollama_is_ready = test_ollama_connection()

Status code: 200
Ollama is working.


Format evidence for prompt

In [38]:
def format_evidence_for_prompt(evidence_df, max_chars_per_chunk=900):
    """
    Format retrieved evidence into a clean prompt section.

    Smaller evidence chunks make local Ollama generation much faster.
    """

    # Return empty string if no evidence exists
    if evidence_df.empty:
        return ""

    # Create empty list for evidence blocks
    evidence_blocks = []

    # Loop through evidence rows
    for index, row in evidence_df.iterrows():

        # Create source number
        source_number = index + 1

        # Shorten chunk text so the prompt does not become too large
        chunk_text = str(row["chunk_text"])[:max_chars_per_chunk]

        # Get citation label
        citation_label = row.get("citation_label", "")

        # Get filing URL
        filing_url = row.get("filing_url", "")

        # Build one evidence block
        evidence_block = f"""
SOURCE {source_number}
Citation: {citation_label}
Filing URL: {filing_url}
Text:
{chunk_text}
"""

        # Store evidence block
        evidence_blocks.append(evidence_block.strip())

    # Join evidence blocks into one evidence section
    formatted_evidence = "\n\n---\n\n".join(evidence_blocks)

    # Return formatted evidence
    return formatted_evidence

#### Build RAG prompt

In [39]:
def build_rag_prompt(question, evidence_df):
    """
    Build a grounded RAG prompt using the user question and retrieved evidence.
    """

    # Format evidence
    formatted_evidence = format_evidence_for_prompt(evidence_df)

    # Build prompt
    prompt = f"""
You are RiskRadar AI, a financial risk analysis assistant.

You must answer using only the SEC filing evidence provided below.

Rules:
1. Do not use outside knowledge.
2. Do not make unsupported claims.
3. If the evidence is not enough, say: "The retrieved SEC evidence is not sufficient to answer this."
4. Cite sources using bracketed source numbers like [Source 1], [Source 2].
5. Keep the answer clear, business-focused, and concise.
6. Include a short "Evidence Used" section at the end.

User question:
{question}

Retrieved SEC evidence:
{formatted_evidence}

Answer:
"""

    # Return final prompt
    return prompt.strip()

#### Generate answer with Ollama

In [40]:
def generate_answer_with_ollama(prompt, temperature=0.1, max_tokens=350, timeout_seconds=600):
    """
    Generate an answer using local Ollama.

    This version uses:
    - fewer output tokens
    - longer timeout
    - safer error handling
    """

    # Create Ollama request payload
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": max_tokens
        }
    }

    # Try to generate answer
    try:
        # Send request to Ollama
        response = requests.post(
            OLLAMA_URL,
            json=payload,
            timeout=timeout_seconds
        )

        # Raise error if request fails
        response.raise_for_status()

        # Parse JSON response
        result = response.json()

        # Return generated answer
        return result.get("response", "").strip()

    except requests.exceptions.ReadTimeout:
        return (
            "Ollama timed out before finishing the answer. "
            "Use a smaller model, reduce top_k, or reduce max evidence length."
        )

    except Exception as error:
        return f"Ollama generation failed: {error}"

Full RAG answer function

In [41]:
def answer_question_with_rag(
    question,
    ticker=None,
    section_name=None,
    top_k=5,
    candidate_k=30,
    use_reranker=True
):
    """
    Retrieve evidence and generate a grounded answer.
    """

    # Start timer
    start_time = time.time()

    # Retrieve evidence
    evidence_df = retrieve_evidence(
        query=question,
        top_k=top_k,
        candidate_k=candidate_k,
        ticker=ticker,
        section_name=section_name,
        use_reranker=use_reranker
    )

    # Stop if no evidence is found
    if evidence_df.empty:
        return {
            "question": question,
            "answer": "No relevant SEC evidence was retrieved.",
            "evidence": evidence_df,
            "prompt": None,
            "total_time_seconds": None
        }

    # Build grounded prompt
    prompt = build_rag_prompt(
        question=question,
        evidence_df=evidence_df
    )

    # Generate answer with Ollama
    answer = generate_answer_with_ollama(prompt)

    # End timer
    end_time = time.time()

    # Return answer package
    return {
        "question": question,
        "answer": answer,
        "evidence": evidence_df,
        "prompt": prompt,
        "total_time_seconds": round(end_time - start_time, 2)
    }

Test one RAG answer

In [42]:
# Define test question
test_question = "What AI and competition risks does NVIDIA mention?"

# Generate RAG answer with fewer evidence chunks
# This is faster for local Ollama
rag_result = answer_question_with_rag(
    question=test_question,
    ticker="NVDA",
    section_name="item_1a_risk_factors",
    top_k=3,
    candidate_k=15,
    use_reranker=True
)

# Print question and answer
print("Question:")
print(rag_result["question"])

print("\nAnswer:")
print(rag_result["answer"])

print("\nTotal time:")
print(rag_result["total_time_seconds"], "seconds")

Question:
What AI and competition risks does NVIDIA mention?

Answer:
NVIDIA mentions the following AI and competition risks:

* The risk of regulatory scrutiny from governments and regulators regarding their sales of GPUs and other NVIDIA products, foundation models, and investments in companies developing foundation models [Source 1].
* The risk that compliance with data privacy requirements, competition and antitrust laws, and other regulations could be onerous and expensive, impacting their competitive position and business operations [Source 2].
* The risk that the economic outcomes of licensing technology may not achieve desired results or customer adoption, potentially negatively impacting their business, operating results, and financial condition [Source 3].

Evidence Used:
[Source 1: NVDA 2026-02-25 10-K, item_1a_risk_factors, chunk 41]
[Source 2: NVDA 2026-02-25 10-K, item_1a_risk_factors, chunk 40]
[Source 3: NVDA 2026-02-25 10-K, item_1a_risk_factors, chunk 8]

Total time:


Inspect evidence used

In [43]:
# Display evidence used for the answer
rag_result["evidence"][
    [
        "rerank_rank",
        "rerank_score",
        "ticker",
        "section_name",
        "citation_label",
        "filing_url"
    ]
]

,rerank_rank,rerank_score,ticker,section_name,citation_label,filing_url
0,1,1.623981,NVDA,item_1a_risk_factors,"NVDA 2026-02-25 10-K, item_1a_risk_factors, ch...",https://www.sec.gov/Archives/edgar/data/104581...
1,2,1.410849,NVDA,item_1a_risk_factors,"NVDA 2026-02-25 10-K, item_1a_risk_factors, ch...",https://www.sec.gov/Archives/edgar/data/104581...
2,3,0.331172,NVDA,item_1a_risk_factors,"NVDA 2026-02-25 10-K, item_1a_risk_factors, ch...",https://www.sec.gov/Archives/edgar/data/104581...


Preview evidence text

In [44]:
def preview_evidence(evidence_df, text_chars=700):
    """
    Print retrieved evidence in a readable format.
    """

    # Handle empty evidence
    if evidence_df.empty:
        print("No evidence found.")
        return

    # Loop through evidence rows
    for _, row in evidence_df.iterrows():

        # Print separator
        print("=" * 100)

        # Print citation metadata
        print("Rank:", row.get("rerank_rank", ""))
        print("Ticker:", row.get("ticker", ""))
        print("Section:", row.get("section_name", ""))
        print("Citation:", row.get("citation_label", ""))
        print("Filing URL:", row.get("filing_url", ""))
        print("-" * 100)

        # Print wrapped text preview
        preview_text = str(row["chunk_text"])[:text_chars]
        print(textwrap.fill(preview_text, width=110))
        print()


# Preview evidence used for test answer
preview_evidence(
    evidence_df=rag_result["evidence"],
    text_chars=900
)

Rank: 1
Ticker: NVDA
Section: item_1a_risk_factors
Citation: NVDA 2026-02-25 10-K, item_1a_risk_factors, chunk 41
Filing URL: https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm
----------------------------------------------------------------------------------------------------
for information from competition regulators in the European Union, the United States, the United Kingdom,
China, and South Korea regarding our sales of GPUs and other NVIDIA products, our efforts to allocate supply,
foundation models and our investments, partnerships and other agreements with companies developing foundation
models, the markets in which we compete and our competition, our strategies, roadmaps, and efforts to develop,
market, and sell hardware, software, and system solutions, and our agreements with customers, suppliers, and
partners. We expect to receive additional requests for information in the future. Such requests have been and
are likely to be expensive and b

Run multiple demo questions

## Demo Questions

Now we test several business-focused demo questions.

These examples show that the system can answer company-specific risk questions using real SEC filing evidence.

In [45]:
# Create demo questions for the RAG system
demo_questions = [
    {
        "question": "What AI and competition risks does NVIDIA mention?",
        "ticker": "NVDA",
        "section_name": "item_1a_risk_factors"
    },
    {
        "question": "What cybersecurity risks does Microsoft mention?",
        "ticker": "MSFT",
        "section_name": "item_1a_risk_factors"
    },
    {
        "question": "What supply chain risks does Tesla mention?",
        "ticker": "TSLA",
        "section_name": "item_1a_risk_factors"
    },
    {
        "question": "What competition risks does Apple describe?",
        "ticker": "AAPL",
        "section_name": "item_1a_risk_factors"
    },
    {
        "question": "What semiconductor competition risks does AMD mention?",
        "ticker": "AMD",
        "section_name": "item_1a_risk_factors"
    }
]

# Display demo questions
pd.DataFrame(demo_questions)

,question,ticker,section_name
0,What AI and competition risks does NVIDIA ment...,NVDA,item_1a_risk_factors
1,What cybersecurity risks does Microsoft mention?,MSFT,item_1a_risk_factors
2,What supply chain risks does Tesla mention?,TSLA,item_1a_risk_factors
3,What competition risks does Apple describe?,AAPL,item_1a_risk_factors
4,What semiconductor competition risks does AMD ...,AMD,item_1a_risk_factors


Generate demo answers

In [ ]:
# Create empty list for demo answer records
demo_answer_records = []

# Loop through each demo question
for demo in demo_questions:

    # Print progress
    print("Generating answer for:", demo["question"])

    # Generate answer
    result = answer_question_with_rag(
        question=demo["question"],
        ticker=demo["ticker"],
        section_name=demo["section_name"],
        top_k=3,
        candidate_k=15,
        use_reranker=True
    )

    # Get evidence table
    evidence_df = result["evidence"]

    # Create compact citation list
    citations = (
        evidence_df["citation_label"].tolist()
        if not evidence_df.empty
        else []
    )

    # Store answer record
    demo_answer_records.append({
        "question": result["question"],
        "ticker": demo["ticker"],
        "section_name": demo["section_name"],
        "answer": result["answer"],
        "citations": citations,
        "total_time_seconds": result["total_time_seconds"]
    })

# Convert demo answers to DataFrame
demo_answers_df = pd.DataFrame(demo_answer_records)

# Display demo answers
demo_answers_df

Generating answer for: What AI and competition risks does NVIDIA mention?
Generating answer for: What cybersecurity risks does Microsoft mention?
Generating answer for: What supply chain risks does Tesla mention?
Generating answer for: What competition risks does Apple describe?
Generating answer for: What semiconductor competition risks does AMD mention?


,question,ticker,section_name,answer,citations,total_time_seconds
0,What AI and competition risks does NVIDIA ment...,NVDA,item_1a_risk_factors,"Based on the provided SEC evidence, NVIDIA men...","[NVDA 2026-02-25 10-K, item_1a_risk_factors, c...",74.21
1,What cybersecurity risks does Microsoft mention?,MSFT,item_1a_risk_factors,Microsoft mentions the following cybersecurity...,"[MSFT 2025-07-30 10-K, item_1a_risk_factors, c...",94.54
2,What supply chain risks does Tesla mention?,TSLA,item_1a_risk_factors,"Based on the provided SEC evidence, Tesla ment...","[TSLA 2026-01-29 10-K, item_1a_risk_factors, c...",126.40
3,What competition risks does Apple describe?,AAPL,item_1a_risk_factors,"Based on the provided SEC evidence, Apple desc...","[AAPL 2025-10-31 10-K, item_1a_risk_factors, c...",99.38
4,What semiconductor competition risks does AMD ...,AMD,item_1a_risk_factors,"Based on the provided SEC evidence, AMD mentio...","[AMD 2026-02-04 10-K, item_1a_risk_factors, ch...",106.97


Save demo answers

In [47]:
# Set output path for demo answers
demo_answers_file = PROCESSED_DIR / "sec_10k_rag_demo_answers.csv"

# Save demo answers to CSV
demo_answers_df.to_csv(demo_answers_file, index=False)

# Confirm file was saved
print("Saved demo answers to:", demo_answers_file)

Saved demo answers to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_rag_demo_answers.csv


Create readable markdown report

In [48]:
# Set markdown report path
rag_answer_report_file = REPORTS_DIR / "rag_answer_generation_demo.md"

# Create empty list for markdown sections
report_sections = []

# Add report title
report_sections.append("# RiskRadar AI: RAG Answer Generation Demo\n")

# Loop through demo answers
for _, row in demo_answers_df.iterrows():

    # Add question
    report_sections.append(f"## Question\n{row['question']}\n")

    # Add answer
    report_sections.append(f"## Answer\n{row['answer']}\n")

    # Add citations
    report_sections.append("## Citations\n")

    # Loop through citations
    for citation in row["citations"]:
        report_sections.append(f"- {citation}")

    # Add separator
    report_sections.append("\n---\n")

# Join report sections
report_text = "\n".join(report_sections)

# Save report
rag_answer_report_file.write_text(report_text, encoding="utf-8")

# Confirm file was saved
print("Saved RAG answer report to:", rag_answer_report_file)

Saved RAG answer report to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\reports\rag_answer_generation_demo.md


## Simple Groundedness Check

This is a lightweight first check.

We will verify that each generated answer includes source citations.

Later, notebook 09 will create a stronger RAG evaluation workflow.

#### Check answers include citations

In [49]:
# Create a copy of demo answers for validation
answer_validation_df = demo_answers_df.copy()

# Check whether answer text includes source citation markers
answer_validation_df["has_source_marker"] = answer_validation_df["answer"].str.contains(
    r"\[Source\s+\d+\]",
    regex=True
)

# Count citation labels attached to each answer
answer_validation_df["num_retrieved_citations"] = answer_validation_df["citations"].apply(len)

# Display validation table
answer_validation_df[
    [
        "question",
        "ticker",
        "has_source_marker",
        "num_retrieved_citations",
        "total_time_seconds"
    ]
]

,question,ticker,has_source_marker,num_retrieved_citations,total_time_seconds
0,What AI and competition risks does NVIDIA ment...,NVDA,True,3,74.21
1,What cybersecurity risks does Microsoft mention?,MSFT,True,3,94.54
2,What supply chain risks does Tesla mention?,TSLA,True,3,126.40
3,What competition risks does Apple describe?,AAPL,True,3,99.38
4,What semiconductor competition risks does AMD ...,AMD,True,3,106.97


#### Save answer validation

In [50]:
# Set output path for answer validation
answer_validation_file = PROCESSED_DIR / "sec_10k_rag_answer_validation.csv"

# Save answer validation
answer_validation_df.to_csv(answer_validation_file, index=False)

# Confirm file was saved
print("Saved answer validation to:", answer_validation_file)

Saved answer validation to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_rag_answer_validation.csv


In [51]:
# Create final checkpoint table
rag_answer_checkpoint = pd.DataFrame({
    "output": [
        "Demo RAG answers",
        "RAG answer report",
        "Answer validation"
    ],
    "path": [
        str(demo_answers_file),
        str(rag_answer_report_file),
        str(answer_validation_file)
    ],
    "exists": [
        demo_answers_file.exists(),
        rag_answer_report_file.exists(),
        answer_validation_file.exists()
    ]
})

# Display checkpoint table
rag_answer_checkpoint

,output,path,exists
0,Demo RAG answers,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
1,RAG answer report,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
2,Answer validation,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True


Final validation

In [52]:
# Validate notebook 08 outputs

# Check that all output files exist
print("Demo answers file exists:", demo_answers_file.exists())
print("RAG answer report exists:", rag_answer_report_file.exists())
print("Answer validation file exists:", answer_validation_file.exists())

# Check number of generated answers
print("Number of demo answers:", len(demo_answers_df))

# Check whether answers contain citation markers
print("Answers with source markers:", answer_validation_df["has_source_marker"].sum())

# Stop if no answers were generated
if len(demo_answers_df) == 0:
    raise ValueError("No demo answers were generated.")

# Confirm notebook completed successfully
print("Notebook 08 completed successfully.")

Demo answers file exists: True
RAG answer report exists: True
Answer validation file exists: True
Number of demo answers: 5
Answers with source markers: 5
Notebook 08 completed successfully.


## RAG Answer Generation Conclusion

This notebook turned retrieved SEC filing evidence into grounded RAG answers.

The project now has:

```text
question
→ hybrid retrieval
→ reranking
→ evidence formatting
→ grounded prompt
→ local LLM answer
→ citations
→ saved answer report
```

This is the first full RAG loop in RiskRadar AI.

The next notebook will focus on evaluation.

```text
retrieval quality
+ citation presence
+ answer faithfulness
+ evidence relevance
+ failure cases
```